In [ ]:
# Scenario: AI-Powered Project Tracker (Task-Based)
# Imagine a company has deployed an AI-powered project tracker that processes
# team updates using the Groq API. The workflow is modeled as a graph of states,
# where each project update flows through nodes until the task status is refreshed.

# 1. State Definition
# The assistant maintains a notebook-like state for each project:
# - task → The specific work item or milestone.
# - update_text → The update submitted by the team member.
# - status → The interpreted task status (e.g., in progress, completed, blocked).
# - response → The confirmation generated for the team.
# - updated_by → The team member who submitted the update.
# - timestamp → The time when the update was submitted.
# - log → A history of all updates.

# Example input prompts a user may give:
# 1. Task: Q1 financial report
#    Update: Report draft completed and sent for final review
#
# 2. Task: Mobile app testing
#    Update: Testing is still going on, a few bugs remain
#
# 3. Task: API deployment
#    Update: Deployment is blocked because server credentials are missing

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
from google.colab import userdata
from datetime import datetime

# 1. Define State
class ProjectState(TypedDict):
    task: str
    update_text: str
    status: str
    response: str
    updated_by: str
    timestamp: str
    log: List[Dict[str, str]]

# 2. Define Nodes (functions)

# This node sends the task update to Groq API and asks it to classify the task status.
# Expected output from model: only one of these values
# - in progress
# - completed
# - blocked
def process_update(state: ProjectState):
    task = state["task"]
    update_text = state["update_text"]

    # Fetch Groq API key from Colab secrets.
    # Make sure your Colab secret name is exactly: groq_api_key
    groq_api_key = userdata.get("groq_api_key")

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'groq_api_key'.")

    # Prompt sent to Groq API
    prompt = f"""
You are an AI project tracker.

Your job is to read the task name and team update, then classify the task status.

Allowed status values:
- in progress
- completed
- blocked

Task: {task}
Update: {update_text}

Return only the status value and nothing else.
"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {groq_api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": 0
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()

    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format. Full response: {response_json}")

    status = response_json["choices"][0]["message"]["content"].strip().lower()

    # Safety normalization in case the model returns extra spaces/text
    if "completed" in status:
        status = "completed"
    elif "blocked" in status:
        status = "blocked"
    else:
        status = "in progress"

    return {"status": status}


# This node creates a confirmation response for the team.
def generate_response(state: ProjectState):
    return {
        "response": f"Task '{state['task']}' marked as {state['status']}."
    }


# This node stores the update into the project log/history.
def update_log(state: ProjectState):
    entry = {
        "task": state["task"],
        "update": state["update_text"],
        "status": state["status"],
        "updated_by": state["updated_by"],
        "timestamp": state["timestamp"]
    }

    return {
        "log": state["log"] + [entry]
    }


# 3. Build the Graph
graph = StateGraph(ProjectState)

graph.add_node("process_update", process_update)
graph.add_node("generate_response", generate_response)
graph.add_node("update_log", update_log)

# 4. Add Edges
graph.set_entry_point("process_update")
graph.add_edge("process_update", "generate_response")
graph.add_edge("generate_response", "update_log")
graph.add_edge("update_log", END)

# 5. Example Run
if __name__ == "__main__":
    # Take runtime input from the user
    task_input = input("Enter task name: ")
    update_input = input("Enter task update: ")
    updated_by_input = input("Enter your name: ")

    # Current timestamp
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Initial state
    state = {
        "task": task_input,
        "update_text": update_input,
        "status": "",
        "response": "",
        "updated_by": updated_by_input,
        "timestamp": current_time,
        "log": []
    }

    # Compile and run graph
    app = graph.compile()
    result = app.invoke(state)

    # Output
    print("\nFinal Response:")
    print(result["response"])

    print("\nUpdated Log:")
    for item in result["log"]:
        print(item)

Enter task name: Q1 financial report
Enter task update: Report draft completed and sent for final approval
Enter your name: Aditi

Final Response:
Task 'Q1 financial report' marked as completed.

Updated Log:
{'task': 'Q1 financial report', 'update': 'Report draft completed and sent for final approval', 'status': 'completed', 'updated_by': 'Aditi', 'timestamp': '2026-03-20 06:54:48'}
